# ROUH — Morphokinetic Modeling and Decision-Level Late Fusion

**Research demonstration — not for clinical use.** This notebook builds LSTM/GRU models for a blastocyst-formation proxy, explores SHAP using a separate random forest, and demonstrates weighted late fusion using illustrative (unpaired) probabilities.

**Reproducibility caveat:** The original preprocessing fits MinMaxScaler before the train/test split; this creates data leakage. Reported metrics below are historical notebook outputs (not independently rerun). For robust evaluation, split first, fit the scaler on training data only, and rerun models.

Datasets and model artifacts are not included. Configure `ROUH_ANNOTATIONS_DIR` and optionally `ROUH_PROJECT_DIR` to run.


#1.0 Imports & Setup

##1.1 Mount Drive

In [ ]:
# Optional in Google Colab; not needed when using local files.
# from google.colab import drive
# drive.mount('/content/drive')


##1.2 Import Libraries

In [ ]:
import os
from glob import glob

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    balanced_accuracy_score,
    roc_curve
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    GRU,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping

import joblib

## 1.2 Define Dataset and Output Paths

In [ ]:
from pathlib import Path
import os

# Configure these paths before running. Do not commit private clinical data.
PROJECT_DIR = Path(os.environ.get('ROUH_PROJECT_DIR', '.')).expanduser().resolve()
DATASET_DIR = Path(os.environ.get('ROUH_ANNOTATIONS_DIR', str(PROJECT_DIR / 'data' / 'annotations'))).expanduser().resolve()
MODELS_DIR = PROJECT_DIR / 'outputs' / 'models'
RESULTS_DIR = PROJECT_DIR / 'outputs' / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset exists:', DATASET_DIR.exists())
print('Models directory:', MODELS_DIR)
print('Results directory:', RESULTS_DIR)


# 2. Morphokinetic Dataset Creation

## 2.1 Read Annotation CSV Files

Each embryo has an annotation CSV file containing developmental phase names and their timing information. These files are combined into one structured dataset.

In [ ]:
csv_files = glob(
    str(DATASET_DIR / "*.csv")
)

print("Total CSV files:", len(csv_files))
print("Example files:", [Path(f).name for f in csv_files[:5]])

## 2.2 Define Morphokinetic Phases

The developmental phases used in the morphokinetic dataset are defined. Each phase represents a timing event during embryo development.

In [ ]:
all_phases = [
    "tPB2", "tPNa", "tPNf",
    "t2", "t3", "t4", "t5", "t6", "t7", "t8",
    "t9+", "tM", "tSB", "tB", "tEB", "tHB"
]

print("Total phases:", len(all_phases))
print(all_phases)

## 2.3 Build Morphokinetic Dataset

Each CSV file is converted into one row, where each column represents the start time of a developmental phase.

In [ ]:
all_rows = []

for file in csv_files:

    embryo_name = Path(file).name.replace("_phases.csv", "")

    df = pd.read_csv(
        file,
        header=None,
        names=["phase", "start", "end"]
    )

    row = {"embryo": embryo_name}

    for phase in all_phases:
        row[phase] = np.nan

    for _, r in df.iterrows():

        phase = str(r["phase"]).strip()

        if phase in all_phases:
            row[phase] = r["start"]

    all_rows.append(row)

morpho_df = pd.DataFrame(all_rows)

print("Dataset Shape:", morpho_df.shape)

display(morpho_df.head())

## 2.4 Save Raw Morphokinetic Dataset

The constructed dataset is saved before preprocessing.

In [ ]:
raw_dataset_path = RESULTS_DIR / "morphokinetic_raw_dataset.csv"

morpho_df.to_csv(
    raw_dataset_path,
    index=False
)

print("Raw morphokinetic dataset saved to:", raw_dataset_path)

# 3. Exploratory Data Analysis

This section analyzes the morphokinetic dataset before model training, including dataset size, missing values, stage availability, and feature distributions.

## 3.1 Dataset Overview

In [ ]:
print("Rows and Columns:", morpho_df.shape)

display(morpho_df.head())

morpho_df.info()

## 3.2 Missing Values Analysis

In [ ]:
missing_values = morpho_df.isnull().sum()

display(missing_values)

## 3.3 Missing Values Percentage

In [ ]:
missing_percentage = (
    morpho_df.isnull().mean() * 100
).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage": missing_percentage
})

display(missing_summary)

## 3.4 Development Stage Availability

This section shows how many embryos reached later developmental stages.

In [ ]:
stage_columns = [
    "tHB",
    "tEB",
    "tB",
    "tSB",
    "tM",
    "t9+",
    "t8"
]

stage_counts = {}

for stage in stage_columns:
    stage_counts[stage] = morpho_df[stage].notna().sum()

stage_counts_df = pd.DataFrame({
    "Stage": list(stage_counts.keys()),
    "Embryo Count": list(stage_counts.values())
})

display(stage_counts_df)

## 3.5 Development Stage Availability Plot

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    stage_counts_df["Stage"],
    stage_counts_df["Embryo Count"]
)

plt.title("Development Stage Availability")
plt.xlabel("Development Stage")
plt.ylabel("Number of Embryos")

plt.show()

## 3.6 Statistical Summary of Morphokinetic Features

In [ ]:
feature_columns = [
    "tPB2","tPNa","tPNf",
    "t2","t3","t4",
    "t5","t6","t7",
    "t8","t9+",
    "tM","tSB","tB",
    "tEB","tHB"
]

display(
    morpho_df[feature_columns].describe()
)

## 3.7 Duplicate Record Analysis

In [ ]:
duplicate_count = morpho_df.duplicated().sum()

print(
    "Duplicate Rows:",
    duplicate_count
)

## 3.8 Distribution of Morphokinetic Timings

In [ ]:
morpho_df[
    [
        "tPB2","tPNa","tPNf",
        "t2","t3","t4",
        "t5","t6","t7",
        "t8"
    ]
].hist(
    figsize=(15,10),
    bins=20
)

plt.suptitle(
    "Morphokinetic Feature Distributions",
    fontsize=14
)

plt.tight_layout()
plt.savefig(
    RESULTS_DIR /
    "morphokinetic_feature_distributions.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 3.9 Correlation Analysis

Understanding relationships between developmental timings may provide biological insight and help identify highly related features.

In [ ]:
corr_matrix = morpho_df[
    feature_columns
].corr()

plt.figure(
    figsize=(12,10)
)

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0
)

plt.title(
    "Morphokinetic Feature Correlation Matrix"
)

plt.savefig(
    RESULTS_DIR /
    "feature_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# 4. Data Preprocessing

## 4.1 Create Developmental Outcome Target

The target is derived from annotation availability, not independently verified clinical viability:
- 1 = blastocyst-stage timing (`tB`) recorded
- 0 = blastocyst-stage timing (`tB`) not recorded

This is a blastocyst-formation proxy, not a clinically validated pregnancy or implantation outcome.


In [ ]:
morpho_df["target"] = morpho_df["tB"].notna().astype(int)

display(
    morpho_df["target"].value_counts()
)

## 4.2 Target Distribution Visualization

In [ ]:
target_counts = morpho_df[
    "target"
].value_counts()

plt.figure(
    figsize=(5,4)
)

plt.bar(
    ["Non-Viable","Viable"],
    [
        target_counts[0],
        target_counts[1]
    ]
)

plt.title(
    "Blastocyst Formation Target Distribution"
)

plt.ylabel(
    "Number of Embryos"
)

plt.show()

## 4.3 Select Morphokinetic Features

Only developmental timing features before blastocyst formation are used as model inputs.

The columns tB, tEB, and tHB are excluded because they are directly related to the target or occur after the target stage, which would cause data leakage.

In [ ]:
features = [
    "tPB2",
    "tPNa",
    "tPNf",
    "t2",
    "t3",
    "t4",
    "t5",
    "t6",
    "t7",
    "t8",
    "t9+",
    "tM",
    "tSB"
]

X = morpho_df[features]
y = morpho_df["target"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

## 4.4 Missing Value Handling

Missing morphokinetic timings may indicate that an embryo did not reach a specific developmental stage. Therefore, missing values were encoded using a sentinel value of 999.

This value is outside the normal timing range and allows the model to distinguish between normally observed developmental timings and unobserved developmental events.

In [ ]:
X = X.fillna(999)

print("Remaining Missing Values:")
display(X.isnull().sum())

## 4.5 Feature Scaling

Min-Max normalization is applied to transform all timing features into the same numerical range.

This helps recurrent neural networks converge more effectively during training.

In [ ]:
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

print(
    "Scaled Dataset Shape:",
    X_scaled.shape
)

## 4.6 Save Feature Scaler

The scaler is saved so the same preprocessing can be applied later when the model is connected to the ROUH website API.

In [ ]:
scaler_path = RESULTS_DIR / "morphokinetic_scaler.pkl"

joblib.dump(
    scaler,
    scaler_path
)

print("Scaler saved to:", scaler_path)

## 4.7 Train-Test Split

The dataset is divided into training and testing sets using stratified sampling.

The test set remains unseen until final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## 4.8 Reshape Data for Sequential Models

LSTM and GRU models require input in three dimensions:

(samples, timesteps, features)

In this project, the 13 morphokinetic timings are treated as sequential developmental events.

In [ ]:
X_train_seq = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test_seq = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

print("Training Shape:", X_train_seq.shape)
print("Testing Shape :", X_test_seq.shape)

## 4.9 Target Distribution Analysis

The distribution of blastocyst formation outcomes is examined before model training.

Target:
- 1 = Reached Blastocyst Stage (tB)
- 0 = Did Not Reach Blastocyst Stage

In [ ]:
print("Target Distribution")

display(
    y.value_counts()
)

print()

print(
    y.value_counts(normalize=True) * 100
)

In [ ]:
plt.figure(figsize=(5,4))

sns.countplot(
    x=y
)

plt.title(
    "Embryo Viability Class Distribution"
)

plt.xlabel(
    "Target"
)

plt.ylabel(
    "Count"
)

plt.show()

# 5. LSTM Model Development

Long Short-Term Memory (LSTM) networks are designed to learn temporal dependencies within sequential data.

In this project, morphokinetic developmental timings are treated as ordered biological events, making LSTM a suitable architecture for embryo viability prediction.

## 5.1 Define LSTM Architecture

In [ ]:
lstm_model = Sequential([

    LSTM(
        64,
        input_shape=(13, 1)
    ),

    Dropout(
        0.3
    ),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )

])

lstm_model.summary()

## 5.2 Compile LSTM Model

Binary cross-entropy is used because the task is a binary classification problem.

The Adam optimizer is selected because it provides efficient gradient-based optimization for deep learning models.

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

## 5.3 Early Stopping

Early stopping is applied to prevent overfitting and improve model generalization.

Training stops automatically when validation loss no longer improves, and the best-performing model weights are restored.

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

## 5.4 Train LSTM Model

The model is trained using the training set and validated using 20% of the training data.

Early stopping is used to reduce overfitting and retain the best model.

In [ ]:
history_lstm = lstm_model.fit(

    X_train_seq,
    y_train,

    validation_split=0.20,

    epochs=50,

    batch_size=32,

    callbacks=[
        early_stop
    ],

    verbose=1

)

## 5.5 Training History

Training and validation performance are visualized to assess model learning behavior and detect potential overfitting.

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history_lstm.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_lstm.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("LSTM Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.savefig(
    RESULTS_DIR /
    "lstm_accuracy_curve.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history_lstm.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_lstm.history["val_loss"],
    label="Validation Loss"
)

plt.title("LSTM Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.savefig(
    RESULTS_DIR /
    "lstm_loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history_lstm.history["precision"],
    label="Training Precision"
)

plt.plot(
    history_lstm.history["val_precision"],
    label="Validation Precision"
)

plt.plot(
    history_lstm.history["recall"],
    label="Training Recall"
)

plt.plot(
    history_lstm.history["val_recall"],
    label="Validation Recall"
)

plt.title("LSTM Precision and Recall")

plt.xlabel("Epoch")

plt.ylabel("Score")

plt.legend()
plt.savefig(
    RESULTS_DIR /
    "lstm_precision_recall_curve.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

##5.6 Save Best LSTM Model

In [ ]:
lstm_model.save(
    MODELS_DIR / "best_lstm_tB.keras"
)

print("Model saved successfully")

##6.1 Generate Predictions

In [ ]:
y_prob = lstm_model.predict(
    X_test_seq
).flatten()

y_pred = (
    y_prob >= 0.5
).astype(int)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "Precision:",
    precision_score(y_test, y_pred)
)

print(
    "Recall:",
    recall_score(y_test, y_pred)
)

print(
    "F1 Score:",
    f1_score(y_test, y_pred)
)

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(
    y_test,
    y_prob
)

print(
    "ROC-AUC:",
    auc
)

## Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("LSTM Confusion Matrix")

plt.savefig(
    RESULTS_DIR /
    "lstm_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

##6.5 Specificity and Balanced Accuracy

In [ ]:
from sklearn.metrics import balanced_accuracy_score

TN = cm[0,0]
FP = cm[0,1]
FN = cm[1,0]
TP = cm[1,1]

specificity = TN / (TN + FP)

balanced_acc = balanced_accuracy_score(
    y_test,
    y_pred
)

print(
    "Specificity:",
    round(specificity,4)
)

print(
    "Balanced Accuracy:",
    round(balanced_acc,4)
)

##Results of LSTM Summary

In [ ]:
lstm_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Specificity",
        "Balanced Accuracy",
        "ROC-AUC"
    ],
    "Value": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        specificity,
        balanced_acc,
        roc_auc_score(y_test, y_prob)
    ]
})

display(lstm_results)

# 7. GRU Model Development

Gated Recurrent Units (GRUs) are recurrent neural networks designed to capture sequential dependencies while using fewer parameters than LSTM networks.

GRU models are evaluated to compare their effectiveness in predicting blastocyst formation from morphokinetic developmental timings.

##7.1

In [ ]:
gru_model = Sequential([

    GRU(
        64,
        input_shape=(13, 1)
    ),

    Dropout(
        0.3
    ),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )

])

gru_model.summary()

## 7.2 Compile GRU Model

The GRU model is compiled using binary cross-entropy loss and the Adam optimizer.

Accuracy, precision, and recall are monitored during training.

In [ ]:
gru_model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]

)

## 7.3 Train GRU Model

The GRU model is trained using the same data split and training strategy used for the LSTM model to ensure a fair comparison.

In [ ]:
history_gru = gru_model.fit(

    X_train_seq,
    y_train,

    validation_split=0.20,

    epochs=50,

    batch_size=32,

    callbacks=[
        early_stop
    ],

    verbose=1

)

##7.4 Generate Predictions

In [ ]:
y_prob_gru = gru_model.predict(
    X_test_seq
).flatten()

y_pred_gru = (
    y_prob_gru >= 0.5
).astype(int)

##7.5 Calculate metrics

In [ ]:
gru_accuracy = accuracy_score(
    y_test,
    y_pred_gru
)

gru_precision = precision_score(
    y_test,
    y_pred_gru
)

gru_recall = recall_score(
    y_test,
    y_pred_gru
)

gru_f1 = f1_score(
    y_test,
    y_pred_gru
)

gru_auc = roc_auc_score(
    y_test,
    y_prob_gru
)

print("Accuracy:", gru_accuracy)
print("Precision:", gru_precision)
print("Recall:", gru_recall)
print("F1 Score:", gru_f1)
print("ROC-AUC:", gru_auc)

##7.6 Confusion Matrix

In [ ]:
cm_gru = confusion_matrix(
    y_test,
    y_pred_gru
)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm_gru,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("GRU Confusion Matrix")

plt.savefig(
    RESULTS_DIR /
    "gru_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

##Models Comparison

In [ ]:
# Build the comparison from predictions rather than typing metrics manually.
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'LSTM': [accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, zero_division=0),
             recall_score(y_test, y_pred, zero_division=0), f1_score(y_test, y_pred, zero_division=0),
             roc_auc_score(y_test, y_prob)],
    'GRU': [accuracy_score(y_test, y_pred_gru), precision_score(y_test, y_pred_gru, zero_division=0),
            recall_score(y_test, y_pred_gru, zero_division=0), f1_score(y_test, y_pred_gru, zero_division=0),
            roc_auc_score(y_test, y_prob_gru)]
})
display(comparison_df)


##Save best gru model

In [ ]:
gru_model.save(
    MODELS_DIR /
    "best_gru_tB.keras"
)

#8 Explainable AI Using SHAP

In [ ]:
import shap

background = X_train[:100]

from sklearn.ensemble import RandomForestClassifier

rf_explainer = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_explainer.fit(
    X_train,
    y_train
)

In [ ]:
explainer = shap.TreeExplainer(
    rf_explainer
)

shap_values = explainer.shap_values(
    X_test
)

In [ ]:
plt.figure(figsize=(10,8))

shap.summary_plot(
    shap_values[:, :, 1],
    X_test,
    feature_names=features,
    show=False
)

plt.savefig(
    RESULTS_DIR / "shap_beeswarm.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()

In [ ]:
shap.summary_plot(
    shap_values[:, :, 1],
    X_test,
    feature_names=features,
    plot_type="bar"
)
plt.savefig(
    RESULTS_DIR /
    "shap_bar_plot.png",
    dpi=300,
    bbox_inches="tight"
)

# 15. Decision-Level Late Fusion Framework

The proposed ROUH system integrates predictions from two complementary AI models:

1. Image-Based Embryo Assessment Model (EfficientNetB0)
2. Morphokinetic Development Prediction Model (LSTM)

Due to the absence of a multimodal dataset containing both image and morphokinetic information for the same embryos, decision-level fusion is used instead of direct multimodal training.

The final embryo assessment score is generated by combining the output probabilities from both models using weighted averaging.

## 15.2 Fusion Strategy

A weighted averaging approach is used to combine model predictions.

Image-based assessment is assigned a higher contribution because it directly evaluates embryo morphology.

Morphokinetic assessment provides complementary developmental information.

In [ ]:
IMAGE_WEIGHT = 0.6
MORPHO_WEIGHT = 0.4

## 15.3 Fusion Function

In [ ]:
def late_fusion_score(
    image_probability,
    morpho_probability
):

    final_score = (
        IMAGE_WEIGHT * image_probability
        +
        MORPHO_WEIGHT * morpho_probability
    )

    return final_score

## 15.4 Fusion Demonstration Example

The following examples are illustrative only. They are not real embryo cases because the image dataset and morphokinetic dataset are not matched embryo-by-embryo.

This section demonstrates how the proposed decision-level fusion function would work inside the ROUH system when both model probabilities are available.

In [ ]:
fusion_examples = pd.DataFrame({
    "Example": ["Embryo 1", "Embryo 2", "Embryo 3"],
    "Image_Probability": [0.82, 0.45, 0.70],
    "Morphokinetic_Probability": [0.76, 0.60, 0.88]
})

fusion_examples["Final_Fusion_Score"] = fusion_examples.apply(
    lambda row: late_fusion_score(
        row["Image_Probability"],
        row["Morphokinetic_Probability"]
    ),
    axis=1
)

display(fusion_examples)

## Save

In [ ]:
fusion_config = {
    "image_weight": IMAGE_WEIGHT,
    "morphokinetic_weight": MORPHO_WEIGHT,
    "method": "weighted_average",
    "note": "Conceptual decision-level fusion because datasets are not paired."
}

import json

with open(RESULTS_DIR / "fusion_config.json", "w") as f:
    json.dump(fusion_config, f, indent=4)

fusion_examples.to_csv(
    RESULTS_DIR / "fusion_demonstration_examples.csv",
    index=False
)

print("Fusion configuration and demonstration examples saved.")

## 15.6 Deployment Note

For the ROUH website, the hard-coded demonstration values will not be used.

Instead, the website backend will generate:

- image_probability from the EfficientNetB0 image model
- morphokinetic_probability from the LSTM morphokinetic model

These two probabilities will then be passed into the late_fusion_score function to produce the final ROUH assessment score.